<a href="https://colab.research.google.com/github/VISHNUVARDHAN2730/NLP-4080/blob/main/2403A54080_NLP_LAB_Assignment_15_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Import Libraries**

In [ ]:
import tensorflow as tf
import tensorflow_hub as hub
import tensorflow_text as text
import numpy as np

# **Load ELMO Model**

In [ ]:
elmo = hub.load("https://tfhub.dev/google/elmo/3")

# **Text Corpus**

In [ ]:
sentences = [
    "The bat flew out of the cave at night",
    "He hit the ball with a cricket bat"
]

# **Generate Embeddings**

In [ ]:
embeddings = elmo.signatures['default'](tf.constant(sentences))['elmo']
print(embeddings.shape)

(2, 9, 1024)


# **Inspect Word Embedding (“bat”)**

In [ ]:
# Convert sentences into tokens
tokenized = [sentence.split() for sentence in sentences]

# Find index of "bank"
idx1 = tokenized[0].index("bat")
idx2 = tokenized[1].index("bat")

# Extract embeddings
bank_emb_1 = embeddings[0][idx1]
bank_emb_2 = embeddings[1][idx2]

print("First 10 values (Sentence 1):", bank_emb_1[:10])
print("First 10 values (Sentence 2):", bank_emb_2[:10])

First 10 values (Sentence 1): tf.Tensor(
[-0.45903727 -0.27516434 -0.26348162 -0.12380227 -0.11530864 -0.45880768
 -0.12675564 -0.12455821 -0.2630463  -0.1839562 ], shape=(10,), dtype=float32)
First 10 values (Sentence 2): tf.Tensor(
[-0.24567907 -0.3350459  -0.20816092 -0.32535243 -0.21820855 -0.02323475
 -0.22867157  0.45242956  0.3082872  -0.09956904], shape=(10,), dtype=float32)


# **Compare Context (Cosine Similarity)**

In [ ]:
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

sim = cosine_similarity(bank_emb_1.numpy(), bank_emb_2.numpy())

print("Similarity between 'bat' meanings:", sim)

Similarity between 'bat' meanings: 0.61797893


# **Task - II**

text corpus=["He ate an apple",
"Apple launched a new laptop"]

# **BERT Model Implementation**

In [ ]:
sentences = [
    "He ate an apple",
    "Apple launched a new laptop"
]

| Component           | Role                          |
| ------------------- | ----------------------------- |
| Preprocessing model | Converts text → numbers       |
| BERT encoder        | Converts numbers → embeddings |


**Preprocessing Model**
Lowercasing
Tokenization
Add Special Tokens
Convert words → IDs
Create inputs required by BERT

(batch_size, sequence_length, 768)

In [ ]:
# Preprocessing model
preprocess = hub.load("https://tfhub.dev/tensorflow/bert_en_uncased_preprocess/3")

# BERT encoder
bert_model = hub.load("https://tfhub.dev/tensorflow/bert_en_uncased_L-12_H-768_A-12/3")

# **Preprocess Input**

In [ ]:
inputs = preprocess(sentences)
print(inputs.keys())

dict_keys(['input_type_ids', 'input_mask', 'input_word_ids'])


# **Generate Embeddings**

In [ ]:
outputs = bert_model(inputs)

# Word-level embeddings
embeddings = outputs['sequence_output']

print("Embedding shape:", embeddings.shape)

Embedding shape: (2, 128, 768)


# **Get Tokens (Important Step)**

In [ ]:
# Use tokenizer from preprocess model
tokenizer = hub.KerasLayer("https://tfhub.dev/tensorflow/bert_en_uncased_preprocess/3")

# Tokenized words (for visualization)
tokens = preprocess(sentences)['input_word_ids']

print(tokens)

tf.Tensor(
[[  101  2002  8823  2019  6207   102     0     0     0     0     0     0
      0     0     0     0     0     0     0     0     0     0     0     0
      0     0     0     0     0     0     0     0     0     0     0     0
      0     0     0     0     0     0     0     0     0     0     0     0
      0     0     0     0     0     0     0     0     0     0     0     0
      0     0     0     0     0     0     0     0     0     0     0     0
      0     0     0     0     0     0     0     0     0     0     0     0
      0     0     0     0     0     0     0     0     0     0     0     0
      0     0     0     0     0     0     0     0     0     0     0     0
      0     0     0     0     0     0     0     0     0     0     0     0
      0     0     0     0     0     0     0     0]
 [  101  6207  3390  1037  2047 12191   102     0     0     0     0     0
      0     0     0     0     0     0     0     0     0     0     0     0
      0     0     0     0     0     0     0     0 

# **Extract bat Embeddings**

In [ ]:
apple_emb_1 = embeddings[0][4]  # "apple" in first sentence
apple_emb_2 = embeddings[0][1]  # "Apple" in second sentence
print("First 10 values (Sentence 1):", apple_emb_1[:10])
print("First 10 values (Sentence 2):", apple_emb_2[:10])

First 10 values (Sentence 1): tf.Tensor(
[-0.4830173   0.07456695 -0.8231044  -0.1901643   0.20118386  0.7980001
 -0.09152785  0.96823555 -0.40703183 -0.38922885], shape=(10,), dtype=float32)
First 10 values (Sentence 2): tf.Tensor(
[ 0.13836057  0.3112528   0.15989228  0.19472355  0.04858243  0.5921318
 -0.5515215   0.963751    0.0735987  -0.5836813 ], shape=(10,), dtype=float32)


# **Cosine Similarity**

In [ ]:
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

sim = cosine_similarity(bat_emb_1.numpy(), bat_emb_2.numpy())

print("Similarity between 'apple' meanings:", sim)

Similarity between 'apple' meanings: 0.416125


# **Text Classification Using ELMO+Naive Bayes**

# **Import Libraries**

In [ ]:
import tensorflow as tf
import tensorflow_hub as hub
import numpy as np

from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# **Text Corpus**

In [ ]:
sentences = [
    "Congratulations you won a cash prize",
    "Call me when you are free",
    "Limited time offer buy now",
    "Let's have lunch together",
    "Earn money from home easily",
    "Did you finish your homework"
]

labels = [1, 0, 1, 0, 1, 0]

# **Load ELMo Model**

In [ ]:
elmo = hub.load("https://tfhub.dev/google/elmo/3")

# **Generate ELMo Embeddings**

In [ ]:
embeddings = elmo.signatures['default'](tf.constant(sentences))['elmo']

print("Shape:", embeddings.shape)

Shape: (6, 6, 1024)


# **Convert to Sentence Embeddings**

In [ ]:
sentence_embeddings = tf.reduce_mean(embeddings, axis=1)

X = sentence_embeddings.numpy()
y = np.array(labels)

print("Feature shape:", X.shape)

Feature shape: (6, 1024)


# **Train-Test Split**

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# **Train Model**

In [ ]:
model = GaussianNB()
model.fit(X_train, y_train)

GaussianNB()

# **Model Testing**

In [ ]:
y_pred = model.predict(X_test)

print("Predictions:", y_pred)
print("Actual:", y_test)

Predictions: [1 1]
Actual: [1 0]


# **Model Evaluation**

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

Accuracy: 0.5


# **Prediction on New Text**

In [ ]:
new_sentence = ["Congratulations! You won a free ticket"]

new_emb = elmo.signatures['default'](tf.constant(new_sentence))['elmo']
new_emb = tf.reduce_mean(new_emb, axis=1)

prediction = model.predict(new_emb.numpy())

print("Prediction:", "Spam" if prediction[0] == 1 else "Not Spam")

Prediction: Spam


# **Text Classification using BERT+NB**

In [ ]:
preprocess = hub.load("https://tfhub.dev/tensorflow/bert_en_uncased_preprocess/3")
bert_model = hub.load("https://tfhub.dev/tensorflow/bert_en_uncased_L-12_H-768_A-12/3")

In [ ]:
bert_inputs = preprocess(sentences)

In [ ]:
bert_outputs = bert_model(bert_inputs)

# Use sentence embedding ([CLS])
bert_features = bert_outputs['pooled_output'].numpy()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    bert_features, labels, test_size=0.2, random_state=42
)

nb_bert = GaussianNB()
nb_bert.fit(X_train, y_train)

y_pred_bert = nb_bert.predict(X_test)
acc_bert = accuracy_score(y_test, y_pred_bert)

print("BERT Accuracy:", acc_bert)

BERT Accuracy: 0.0
